In [17]:

PDF_URL = "https://arxiv.org/pdf/2602.15705.pdf"
OUT_FILE = "paper.md"  

In [18]:
def download_file(url: str, out_path: str, timeout: int = 180) -> None:
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(1024 * 128):
                if chunk:
                    f.write(chunk)

def poll_batch(batch_id: str, poll_seconds: int = 5, timeout_seconds: int = 900) -> dict:
    info_url = f"{BASE}/v1/batches/{batch_id}"
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        r = requests.get(info_url, headers=HEADERS_AUTH, timeout=30)
        r.raise_for_status()
        info = r.json()
        status = info.get("status")
        print("status:", status)
        if status in ("completed", "failed"):
            return info
        time.sleep(poll_seconds)
    raise TimeoutError("Batch did not complete within timeout_seconds")

def save_markdown_from_retrieve(retrieve_id: str, out_file: str) -> None:
    r = requests.get(
        f"{BASE}/v1/retrieve",
        headers=HEADERS_AUTH,
        params={"retrieve_id": retrieve_id, "formats": ["markdown"]},
        timeout=180,
    )
    r.raise_for_status()
    data = r.json()
    result = data.get("result", {})

    md = result.get("markdown_content")
    hosted = result.get("markdown_hosted_url")

    if result.get("size_exceeded") and hosted:
        download_file(hosted, out_file)
        print("Saved (hosted):", out_file)
        return

    if md:
        with open(out_file, "w", encoding="utf-8") as f:
            f.write(md)
        print("Saved (inline):", out_file, "| chars:", len(md))
        return

    raise RuntimeError(f"No markdown returned. Keys: {list(result.keys())}")


In [23]:
create_url = f"{BASE}/v1/batches"

payload = {
    "batch_array": [
        {"custom_id": "paper_1", "url": PDF_URL}
    ],
    "formats": ["markdown"],
}

resp = requests.post(create_url, json=payload, headers=headers, timeout=60)
print("status:", resp.status_code)
print(resp.text[:1000])
resp.raise_for_status()

batch_id = resp.json()["id"]
batch_id


status: 200
{"id":"batch_x1c39epdr9","object":"batch","status":"in_progress","created":1771400348212,"total_urls":1,"completed_urls":0,"number_retried":0,"batch_parser":"none","parser":"none","batch_country":"RANDOM","country":"RANDOM","start_date":"2026-02-18","metadata":{}}


'batch_x1c39epdr9'

In [25]:
import time, requests

BASE = "https://api.olostep.com"
HEADERS_AUTH = {"Authorization": f"Bearer {API_KEY}"}

batch_id = "batch_x1c39epdr9"

info_url = f"{BASE}/v1/batches/{batch_id}"

while True:
    r = requests.get(info_url, headers=HEADERS_AUTH, timeout=30)
    r.raise_for_status()
    info = r.json()
    status = info.get("status")
    print("status:", status)
    if status in ("completed", "failed"):
        break
    time.sleep(5)

if status == "completed":
    retrieve_id = info.get("retrieve_id")
    save_markdown_from_retrieve(retrieve_id, OUT_FILE)


status: completed


HTTPError: 400 Client Error: Bad Request for url: https://api.olostep.com/v1/retrieve?formats=markdown